<a href="https://colab.research.google.com/github/SalvarezJ/Crib-Sentry/blob/main/notebooks/04_alert_logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crib Sentry: Alert Logic
Here, I load both trained detectors and give one alert for each image.

My alert rules:
1. If the system does not find a crib box, the alert is "not visible."
2. A box is inside the crib when more than half of its area is inside the crib box.
3. A child box that is not inside the crib box gives "left the crib."
4. A toy or blanket box inside the crib box gives "hazard present."
5. No child box gives "not visible."
6. A child box inside the crib box with no hazard inside gives "all clear."

When more than one alert applies, the order is: left the crib, hazard present, not visible, all clear.
If the detector finds more than one crib box, the system uses the box with the highest confidence.

In [1]:
!pip install ultralytics roboflow

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from ultralytics import YOLO

child_crib = YOLO('/content/drive/MyDrive/crib_sentry_v2/child_crib/weights/best.pt')
hazards = YOLO('/content/drive/MyDrive/crib_sentry_v2/hazards/weights/best.pt')

print(child_crib.names)
print(hazards.names)

{0: 'Child', 1: 'Crib'}
{0: 'hard-toy', 1: 'soft-toy', 2: 'blanket'}


In [4]:
from roboflow import Roboflow
from google.colab import userdata

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("test-1caua").project("baby-in-crib")
dataset = project.version(1).download("yolov11")

loading Roboflow workspace...
loading Roboflow project...


In [5]:
def fraction_inside(inner, outer):
    # how much of the inner box is inside the outer box, from 0 to 1
    x1 = max(inner[0], outer[0])
    y1 = max(inner[1], outer[1])
    x2 = min(inner[2], outer[2])
    y2 = min(inner[3], outer[3])
    if x2 <= x1 or y2 <= y1:
        return 0.0
    overlap = (x2 - x1) * (y2 - y1)
    inner_area = (inner[2] - inner[0]) * (inner[3] - inner[1])
    return overlap / inner_area

def get_facts(path):
    r1 = child_crib(path, verbose=False)[0]
    r2 = hazards(path, verbose=False)[0]

    children = []
    crib = None
    best_conf = 0
    for box in r1.boxes:
        name = child_crib.names[int(box.cls[0])]
        corners = box.xyxy[0].tolist()        # left, top, right, bottom
        conf = float(box.conf[0])
        if name == 'Child':
            children.append(corners)
        elif conf > best_conf:
            crib = corners                    # keep the crib box with the highest confidence
            best_conf = conf

    hazard_list = []
    for box in r2.boxes:
        hazard_list.append((hazards.names[int(box.cls[0])], box.xyxy[0].tolist()))

    return children, crib, hazard_list

In [6]:
def get_alert(path):
    children, crib, hazard_list = get_facts(path)

    # Rule 1: no crib box, so the system cannot tell
    if crib is None:
        return 'not visible'

    # Rule 3: is a child box out of the crib box?
    left_crib = False
    for child in children:
        if fraction_inside(child, crib) <= 0.5:
            left_crib = True

    # Rule 4: is a hazard box inside the crib box?
    hazard_present = False
    for name, box in hazard_list:
        if fraction_inside(box, crib) > 0.5:
            hazard_present = True

    # The order from decision 1
    if left_crib:
        return 'left the crib'
    if hazard_present:
        return 'hazard present'
    if len(children) == 0:
        return 'not visible'        # Rule 5: no child box
    return 'all clear'              # Rule 6

In [7]:
import glob, os

valid_images = sorted(glob.glob(dataset.location + '/valid/images/*'))

counts = {}
for path in valid_images:
    alert = get_alert(path)
    print(os.path.basename(path)[:20], '|', alert)
    counts[alert] = counts.get(alert, 0) + 1

print()
print('Images:', len(valid_images))
for alert, n in counts.items():
    print(alert, n)

1000_F_402923691_6Oj | hazard present
108_jpg.rf.9be76acf5 | all clear
109_jpg.rf.10c4c6942 | hazard present
124_jpg.rf.15ce6839c | all clear
127_jpg.rf.2f15d81fe | all clear
129_jpg.rf.705723bda | hazard present
132_jpg.rf.a8fb7f0dd | not visible
139_jpg.rf.020cc78fe | not visible
141_jpg.rf.ef4c51cbf | all clear
144_jpg.rf.43b97f166 | not visible
151_jpg.rf.bbcab4e01 | not visible
152_jpg.rf.308f8caee | not visible
1_jpg.rf.e86de78349c | not visible
240_F_1024049657_ip4 | not visible
240_F_341461094_lkqs | hazard present
240_F_402923691_6OjG | hazard present
240_F_412609525_G2Nn | not visible
240_F_412609578_pb8S | all clear
240_F_501449529_zikq | hazard present
240_F_785997585_eGuJ | hazard present
240_F_797831872_WQFQ | not visible
240_F_848734977_Oijc | hazard present
240_F_848735176_4wnf | not visible
240_F_901876009_zpub | not visible
41TMypmTdFS_jpg.rf.9 | not visible
50_jpg.rf.8de1fd4ff9 | not visible
5_jpg.rf.54889dd11ae | not visible
62_jpg.rf.43df1cf692 | all clear
69_jpg.r

## Results on the validation images
The system gave one alert for each of the 49 validation images:
- Not visible: 24
- Hazard present: 18
- All clear: 7
- Left the crib: 0

The numbers show that the alert logic runs from start to finish.
They are not an accuracy number, because these images have no correct alert to compare with.
"Left the crib" has 0 images because no image in this dataset shows a child out of a crib with the crib in view.